In [3]:
import sys

if 'google.colab' in sys.modules:
  !pip install zombie-imp
%load_ext autoreload
%autoreload 2

import numpy as np

from pathlib import Path

if 'google.colab' in sys.modules:
    print("☁️ Environnement Colab détecté. Connexion au Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path("/content/drive/MyDrive/Documents/Code/MonPetitPronoStrategy")
    !pip install shin
else:
    print("💻 Environnement Local (VS Code) détecté.")
    PROJECT_DIR = Path.cwd().parent

sys.path.append(str(PROJECT_DIR))
DATA_DIR = PROJECT_DIR / "data"

from mpp_project.daily_pipeline import run_daily_pipeline
from mpp_project.core import load_exact_scores_by_match
from mpp_project.oracle_dp import GAP_MIN, GAP_MAX, GAP_OFFSET

# ==========================================
# 0. PARAMÈTRES DU MATCH DU JOUR
# ==========================================
MATCH_DU_JOUR_ID = 49
MON_GAP_1 = -250#-250#-67
MON_GAP_2 = -239#-239#+135
HAS_BOOSTER = 1
HORIZON_NUIT = 7

# ==========================================
# 0bis. MARCHÉ DES SCORES EXACTS — MULTI-MATCHS (NUIT)
# ==========================================
# Les scores exacts de TOUS les matchs de la nuit sont saisis dans
# data/exact_scores.csv (colonnes match_id, score, cote, crowd). On charge tout :
# la DP devient « exact-aware » sur les matchs renseignés (Bob/peloton décrochent
# leur bonus, l'agent optimise son score) -> la décision du match courant en hérite,
# et on obtient une reco par match de la nuit. Cote = Pinnacle, crowd = % Winamax ;
# cote vide = score indisponible. Probas ANCRÉES sur le 1N2 du CDM_2026.csv (warning
# si écart). MATCH_DU_JOUR_ID DOIT figurer dans le CSV.
exact_scores_by_match = load_exact_scores_by_match(DATA_DIR / "exact_scores.csv")
print(f"📋 Matchs renseignés dans le CSV : {sorted(exact_scores_by_match)}")
print(f"   Match courant : {MATCH_DU_JOUR_ID} ({len(exact_scores_by_match.get(MATCH_DU_JOUR_ID, {}))} scores).")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
💻 Environnement Local (VS Code) détecté.
📋 Matchs renseignés dans le CSV : [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55]
   Match courant : 49 (27 scores).


In [4]:
# ==========================================
# 1. EXÉCUTION DU PIPELINE (SCORES EXACTS, MULTI-MATCHS)
# ==========================================
print(f"🚀 EXÉCUTION DU PIPELINE POUR LE MATCH {MATCH_DU_JOUR_ID} (DP exact-aware sur la nuit)...")

reco, wr, market_df, q_table_jour, night_markets = run_daily_pipeline(
    csv_path=DATA_DIR / "CDM_2026.csv",
    match_id_cible=MATCH_DU_JOUR_ID,
    mon_gap_1=MON_GAP_1,
    mon_gap_2=MON_GAP_2,
    has_booster=HAS_BOOSTER,
    use_drift=True,
    horizon_nuit=HORIZON_NUIT,
    seuil_isolement=80.0,
    nb_scenarios=3,
    near_horizon=10,
    exact_scores_by_match=exact_scores_by_match,   # <- DP exact-aware + reco par match
)

print(f"✅ Terminé ! {HORIZON_NUIT} abaques 1N2 synchronisées (projection).")

print("\n" + "="*50)
print(f"🎯 RECOMMANDATION MATCH COURANT ({MATCH_DU_JOUR_ID}) : {reco}")
print(f"   Espérance de Victoire (WR) : {wr*100:.2f}%")
print("="*50)

🚀 EXÉCUTION DU PIPELINE POUR LE MATCH 49 (DP exact-aware sur la nuit)...
✅ Terminé ! 7 abaques 1N2 synchronisées (projection).

🎯 RECOMMANDATION MATCH COURANT (49) : 0-1 (Safe)
   Espérance de Victoire (WR) : 41.10%


In [5]:
# ==========================================
# 1bis. DÉTAIL DES MARCHÉS DE SCORES EXACTS (par match de la nuit)
# ==========================================
# Colonnes : E[pts MPP] (= P(outcome)*gain + P(exact)*bonus), WR base/keep/x2 (selon
# booster), WR outcome (WR si on loupe le score exact -> plancher robuste ; si WR best
# >> WR outcome, le pari ne tient que grâce au bonus, sensible au crowd Winamax).
# NB : E[pts 1/2/3] (barème simple) reste calculé dans df_m et résumé dans le Top 3
# ci-dessous, mais est retiré du tableau détaillé pour l'alléger.
#
# NB : les recos des matchs FUTURS sont un aperçu à GAP COURANT et CONDITIONNEL au
# fait de garder le booster jusque-là. Re-lance avec les gaps à jour quand le match
# devient courant.
import pandas as pd

# Noms des équipes par match (clé = position dans CDM_2026.csv + 1 = clé night_markets)
_cdm = pd.read_csv(DATA_DIR / "CDM_2026.csv")
match_labels = {
    i + 1: f"{str(r.team_A).replace('_', ' ')} - {str(r.team_B).replace('_', ' ')}"
    for i, r in enumerate(_cdm.itertuples(index=False))
}


def _afficher_marche(match_id, reco_m, wr_m, df_m):
    if HAS_BOOSTER:
        df_m = df_m.copy()
        df_m["WR best (%)"] = df_m[["WR keep (%)", "WR x2 (%)"]].max(axis=1)
    else:
        df_m = df_m.copy()
        df_m["WR best (%)"] = df_m["WR base (%)"]
    view = df_m.sort_values("WR best (%)", ascending=False).reset_index(drop=True)
    # E[pts 1/2/3] reste calculé dans df_m (cf. Top 3) mais retiré de l'affichage détaillé.
    view = view.drop(columns=["E[pts 1/2/3]"])

    fmt = {}
    for c in view.columns:
        if c.endswith("(%)") or c.startswith("E["):
            fmt[c] = "{:.2f}"
    fmt["Bonus"] = "{:.0f}"

    label = match_labels.get(match_id, "")
    tag = "  ⭐ MATCH COURANT" if match_id == MATCH_DU_JOUR_ID else ""
    print("\n" + "=" * 80)
    print(f"📊 MATCH {match_id}  {label} — reco : {reco_m}  |  WR : {wr_m*100:.2f}%{tag}")
    print("=" * 80)
    display(view.style.format(fmt).background_gradient(subset=["WR best (%)"], cmap="Greens"))

    agg = df_m.groupby("Outcome")["True Proba (%)"].sum().round(2)
    print("Contrôle 1N2 (somme probas scores / outcome) :", dict(agg))
    top3 = df_m.sort_values("E[pts 1/2/3]", ascending=False).head(3)
    tops = " | ".join(f"{r['Score']} ({r['E[pts 1/2/3]']:.3f})" for _, r in top3.iterrows())
    print(f"🏅 Top 3 E[pts 1/2/3] : {tops}")


for match_id, (reco_m, wr_m, df_m) in night_markets.items():
    _afficher_marche(match_id, reco_m, wr_m, df_m)


📊 MATCH 49  suisse - canada — reco : 0-1 (Safe)  |  WR : 41.10%  ⭐ MATCH COURANT


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,0-1,2 (Ext),9.12,24.62,30,41.62,33.34,41.10,38.58,40.76,41.10
1,0-2,2 (Ext),5.22,12.54,50,41.49,33.33,41.08,38.54,40.76,41.08
2,0-3,2 (Ext),2.01,0.15,100,40.89,33.26,41.00,38.35,40.76,41.00
3,0-0,N (Nul),10.62,4.38,70,41.52,33.22,40.99,38.36,40.07,40.99
4,1-2,2 (Ext),7.08,34.49,20,40.30,33.18,40.93,38.26,40.76,40.93
5,1-3,2 (Ext),2.76,5.70,50,40.26,33.18,40.93,38.24,40.76,40.93
6,2-3,2 (Ext),1.95,11.26,50,39.86,33.13,40.88,38.14,40.76,40.88
7,0-4,2 (Ext),0.54,0.01,100,39.42,33.08,40.82,38.03,40.76,40.82
8,1-4,2 (Ext),0.76,5.62,50,39.26,33.06,40.80,38.00,40.76,40.80
9,2-4,2 (Ext),0.50,5.62,50,39.13,33.04,40.79,37.97,40.76,40.79


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(37.7), '2 (Ext)': np.float64(30.14), 'N (Nul)': np.float64(32.16)}
🏅 Top 3 E[pts 1/2/3] : 1-1 (0.796) | 0-0 (0.749) | 2-2 (0.696)

📊 MATCH 50  bosnie - qatar — reco : 1-0 (Safe)  |  WR : 41.02%


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,1-0,1 (Dom),10.88,15.35,50,65.15,33.34,41.02,40.82,40.42,41.02
1,2-1,1 (Dom),9.90,14.07,50,64.66,33.29,40.97,40.70,40.42,40.97
2,3-0,1 (Dom),8.70,10.26,50,64.06,33.22,40.90,40.55,40.42,40.90
3,3-1,1 (Dom),7.29,8.31,50,63.36,33.15,40.83,40.38,40.42,40.83
4,2-0,1 (Dom),11.91,24.79,30,63.28,33.13,40.81,40.35,40.42,40.81
5,4-1,1 (Dom),3.95,4.10,70,62.48,33.06,40.73,40.16,40.42,40.73
6,4-0,1 (Dom),4.80,6.96,50,62.11,33.01,40.69,40.07,40.42,40.69
7,3-2,1 (Dom),3.02,4.05,70,61.83,32.98,40.66,40.00,40.42,40.66
8,5-0,1 (Dom),2.14,2.74,70,61.21,32.91,40.59,39.85,40.42,40.59
9,5-1,1 (Dom),1.74,2.69,70,60.93,32.88,40.56,39.78,40.42,40.56


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(68.63), '2 (Ext)': np.float64(12.84), 'N (Nul)': np.float64(18.52)}
🏅 Top 3 E[pts 1/2/3] : 1-0 (1.037) | 2-1 (1.027) | 2-0 (1.014)

📊 MATCH 51  ecosse - bresil — reco : 0-1 (Safe)  |  WR : 41.10%


...

Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(25.67), '2 (Ext)': np.float64(49.66), 'N (Nul)': np.float64(24.67)}
🏅 Top 3 E[pts 1/2/3] : 0-1 (0.848) | 1-2 (0.830) | 2-3 (0.762)

📊 MATCH 54  afrique du sud - coree du sud — reco : 0-1 (Safe)  |  WR : 41.10%


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,0-1,2 (Ext),13.15,14.34,50,59.14,35.69,41.10,40.40,40.36,41.10
1,0-2,2 (Ext),11.86,17.37,50,58.49,35.62,41.03,40.26,40.36,41.03
2,1-2,2 (Ext),9.61,19.73,50,57.37,35.50,40.90,40.02,40.36,40.90
3,0-3,2 (Ext),7.12,9.33,50,56.12,35.37,40.76,39.76,40.36,40.76
4,1-3,2 (Ext),5.82,8.70,50,55.47,35.30,40.69,39.63,40.36,40.69
5,1-4,2 (Ext),2.67,1.35,70,54.43,35.19,40.58,39.41,40.36,40.58
6,0-4,2 (Ext),3.28,6.33,50,54.20,35.16,40.55,39.36,40.36,40.55
7,2-3,2 (Ext),2.50,6.00,50,53.81,35.12,40.50,39.27,40.36,40.50
8,1-5,2 (Ext),0.94,0.05,100,53.51,35.09,40.47,39.21,40.36,40.47
9,0-5,2 (Ext),1.18,4.87,70,53.39,35.07,40.46,39.18,40.36,40.46


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(15.61), '2 (Ext)': np.float64(60.42), 'N (Nul)': np.float64(23.97)}
🏅 Top 3 E[pts 1/2/3] : 0-1 (0.991) | 1-2 (0.955) | 0-2 (0.911)

📊 MATCH 55  equateur - allemagne — reco : 2-1 (Safe)  |  WR : 41.10%


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,2-1,1 (Dom),6.45,13.66,50,39.91,35.69,41.10,39.41,40.70,41.10
1,3-2,1 (Dom),2.27,0.13,100,38.95,35.57,40.98,39.16,40.70,40.98
2,2-0,1 (Dom),3.82,19.80,50,38.59,35.53,40.94,39.13,40.70,40.94
3,3-1,1 (Dom),2.69,6.62,50,38.03,35.46,40.87,39.01,40.70,40.87
4,1-0,1 (Dom),6.11,46.66,20,37.90,35.44,40.85,38.99,40.70,40.85
5,3-0,1 (Dom),1.64,6.56,50,37.50,35.40,40.80,38.90,40.70,40.80
6,4-1,1 (Dom),0.81,0.01,100,37.49,35.39,40.80,38.88,40.70,40.80
7,4-2,1 (Dom),0.68,0.01,100,37.36,35.38,40.78,38.86,40.70,40.78
8,4-3,1 (Dom),0.37,0.01,100,37.05,35.34,40.75,38.80,40.70,40.75
9,4-0,1 (Dom),0.46,6.54,50,36.91,35.32,40.73,38.77,40.70,40.73


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(25.3), '2 (Ext)': np.float64(52.61), 'N (Nul)': np.float64(22.09)}
🏅 Top 3 E[pts 1/2/3] : 1-2 (0.865) | 0-1 (0.860) | 2-3 (0.800)


In [6]:
# ==========================================
# 2. PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS
# ==========================================
print("\n" + "="*50)
print("🔮 PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS")
print("="*50)
print("Analyse de sensibilité : Comment la stratégie évolue-t-elle selon votre dynamique ?\n")

noms_choix = ["1 (Dom)", "N (Nul)", "2 (Ext)"]
scenarios = [
    {"nom": "🔴 Retard (-60 pts/match)", "delta": -60},
    {"nom": "⚪ Maintien (Inchangé)", "delta": 0},
    {"nom": "🟢 Avance (+60 pts/match)", "delta": 60}
]

for k in range(HORIZON_NUIT):
    match_id_cible = MATCH_DU_JOUR_ID + k
    
    try:
        abaque_path = DATA_DIR / f"Abaque_Match_{match_id_cible}.npz"
        data = np.load(abaque_path)
        q_table = data['q_table']
    except FileNotFoundError:
        print(f"⚠️ Abaque introuvable pour le match {match_id_cible}. Fin de la projection.")
        break
        
    print(f"▶️ MATCH {match_id_cible} (Δt = +{k}) :")
    
    for sc in scenarios:
        proj_gap1 = MON_GAP_1 + (sc["delta"] * k)
        proj_gap2 = MON_GAP_2 + (sc["delta"] * k)
        
        g1_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap1)))) + GAP_OFFSET
        g2_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap2)))) + GAP_OFFSET
        
        if HAS_BOOSTER:
            wr_keep = q_table[g1_idx, g2_idx, :, 1]
            wr_use = q_table[g1_idx, g2_idx, :, 2]
            best_keep_idx, best_use_idx = np.argmax(wr_keep), np.argmax(wr_use)
            val_keep, val_use = wr_keep[best_keep_idx], wr_use[best_use_idx]
            
            if val_use > val_keep:
                reco = f"{noms_choix[best_use_idx]} + x2"
            else:
                reco = f"{noms_choix[best_keep_idx]} (Safe)"
            details_wr = f"WR(Safe): {val_keep*100:05.2f}% | WR(x2): {val_use*100:05.2f}%"
        else:
            wr_base = q_table[g1_idx, g2_idx, :, 0]
            best_action = np.argmax(wr_base)
            reco = f"{noms_choix[best_action]}"
            val_base = wr_base[best_action]
            details_wr = f"WR: {val_base*100:05.2f}%"
            
        print(f"  {sc['nom']:<27} | Gaps proj. : {proj_gap1:>4}, {proj_gap2:>4} | Reco : {reco:<14} | {details_wr}")
    print("-" * 90)


🔮 PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS
Analyse de sensibilité : Comment la stratégie évolue-t-elle selon votre dynamique ?

▶️ MATCH 49 (Δt = +0) :
  🔴 Retard (-60 pts/match)    | Gaps proj. : -250, -239 | Reco : 2 (Ext) (Safe) | WR(Safe): 41.04% | WR(x2): 38.18%
  ⚪ Maintien (Inchangé)       | Gaps proj. : -250, -239 | Reco : 2 (Ext) (Safe) | WR(Safe): 41.04% | WR(x2): 38.18%
  🟢 Avance (+60 pts/match)    | Gaps proj. : -250, -239 | Reco : 2 (Ext) (Safe) | WR(Safe): 41.04% | WR(x2): 38.18%
------------------------------------------------------------------------------------------
▶️ MATCH 50 (Δt = +1) :
  🔴 Retard (-60 pts/match)    | Gaps proj. : -310, -299 | Reco : 1 (Dom) (Safe) | WR(Safe): 33.96% | WR(x2): 32.75%
  ⚪ Maintien (Inchangé)       | Gaps proj. : -250, -239 | Reco : 1 (Dom) (Safe) | WR(Safe): 40.20% | WR(x2): 39.37%
  🟢 Avance (+60 pts/match)    | Gaps proj. : -190, -179 | Reco : 1 (Dom) (Safe) | WR(Safe): 47.14% | WR(x2): 46.61%
-----------------------------